In [2]:
import pickle
import numpy as np
import pandas as pd

import sys

sys.path.append("..")
from cd_zoo.tools.scoring_tools import score
from robustness_experiments.plot import renames

# Generate Causal Rivers Tensor dataset.



Please follow the Readmesteps first before running this code.

In [3]:
def clip_and_normalize(X):
    """
    Clip values to 2 standard deviations and normalize for each method independently.
    X shape: (batch, method, var, var, lag) or (batch, method, var, var)
    Each method slice is normalized independently over all other dimensions.
    """

    X = np.array(X)
    n_methods = X.shape[1]
    X_normalized = np.zeros_like(X)

    # before we do anything, clip occasionnally very large vectors to not skew the training.
    # THis is occasionally happening for models that return coefficients.
    X = X.clip(-10, 10)
    # Process each method independently
    for i in range(n_methods):
        method_slice = X[:, i, ...]
        # Calculate mean and std over all dimensions for this method
        mean = np.mean(method_slice)
        std = np.std(method_slice)
        # Clip to 2 standard deviations
        method_clipped = np.clip(method_slice, mean - 2 * std, mean + 2 * std)
        print("Clipping to:", mean - 2 * std, mean + 2 * std)
        # Normalize the clipped range to [0, 1]
        min_val = np.min(method_clipped)
        max_val = np.max(method_clipped)
        # Avoid division by zero
        range_val = max_val - min_val
        if range_val == 0:
            print("CAREFUL DIVISION BY ZERO IN NORMALIZATION, CHECK YOUR DATA!")
            range_val = 1
        X_normalized[:, i, ...] = (method_clipped - min_val) / range_val
    print(X_normalized.min(), X_normalized.max())
    return X_normalized

In [ ]:
# We need to put the methods in the same order as in the training data. THis is the order in which they are trained-
m_order = ['direct_crosscorr', 'ntsnotears', 'pcmci', 'pcmciplus', 'svarrfci', 'var',"varlingam", "dynotears", "fpcmci"]

['direct_crosscorr', 'ntsnotears', 'pcmci', 'pcmciplus', 'svarrfci', 'var']

In [ ]:
# Here we need to load the predictions from running ./scripts/predict_for_causal_rivers.sh and put them in the same order as the training data.
stack = []
for method in m_order:
    p = f"/path/to/causal_rivers_method_predictions/rivers/{method}/best_run/preds.p"
    b = pickle.load(open(p, "rb"))
    stack.append(b)

In [ ]:
stack = np.stack(stack, axis=1)
stack = clip_and_normalize(stack)
# saving the results for prediction.
pickle.dump(
    stack,
    open(
        "processed_predictions.p",
        "wb",
    ),
)

Clipping to: 0.1476734657457327 1.2436312962316167
Clipping to: -0.40698073571462834 0.6251685808946749
Clipping to: 0.03783181154881854 1.346838671444369
Clipping to: 0.10704855271193636 1.2924781443690487
Clipping to: 0.12630994881107438 1.2728846829204155
Clipping to: -3.392945910275742 5.248005109783078
Clipping to: -1.9501532789332385 2.6570856186611342
Clipping to: -0.46216072424142407 0.628512297556733
Clipping to: -0.559324567370582 0.8523754070542697
0.0 1.0
